# Efficio quick start: ResNet-18 on CIFAR-10 in Colab

This notebook trains ResNet-18 on CIFAR-10 with [Efficio](https://github.com/HuNTer8272/Efficio). It shows the four things most Colab users need:

1. **Mixed precision** chosen for your GPU (fp16 on the free T4).
2. **GPULoader**: CIFAR-10 is small enough to live on the GPU, so batches are built and augmented there instead of on Colab's 2 CPU cores.
3. **A time budget**: the run saves a checkpoint and stops before the budget runs out.
4. **Resume**: run the notebook again (after a disconnect, or in a new session) and training continues from the last checkpoint.

Use a GPU runtime: *Runtime > Change runtime type > T4 GPU*.

In [ ]:
!pip install -q efficio

## 1. Data on the GPU

`GPULoader.from_torchvision` copies the training images to the GPU once (as uint8, about 150 MB for CIFAR-10). Every batch is then cut, flipped and normalized on the GPU with `efficio.gpu_augment`. The test set uses a GPULoader without augmentation.

In [ ]:
import torch
import torchvision
from torch import nn

import efficio
from efficio import GPULoader
from efficio import gpu_augment as A

MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_set = torchvision.datasets.CIFAR10("/content/data", train=True, download=True)
test_set = torchvision.datasets.CIFAR10("/content/data", train=False, download=True)

train_loader = GPULoader.from_torchvision(
    train_set, batch_size=128, shuffle=True, drop_last=True,
    transform=A.Compose([A.RandomCrop(32, padding=4), A.RandomHorizontalFlip(),
                         A.Normalize(MEAN, STD)]))
test_loader = GPULoader.from_torchvision(test_set, batch_size=512, shuffle=False,
                                         transform=A.Normalize(MEAN, STD))
print(len(train_loader), "training batches per epoch")

## 2. Model, optimizer and schedule

ResNet-18 with the usual CIFAR stem (3x3 first convolution, no max-pool), SGD and a one-cycle learning-rate schedule. Nothing here is Efficio-specific.

In [ ]:
EPOCHS = 15

torch.manual_seed(0)
model = torchvision.models.resnet18(num_classes=10)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1,
                                                total_steps=EPOCHS * len(train_loader))
loss_fn = nn.CrossEntropyLoss()

## 3. Start (or resume) the run

`efficio.Run(budget=...)` detects the GPU and picks the precision. `run.prepare(...)` moves the model to the GPU and, if `checkpoint_dir` already holds a checkpoint, restores the model, optimizer, scheduler, RNG state and the exact position in the epoch.

To keep checkpoints across sessions, mount Google Drive and point `checkpoint_dir` to it, for example `/content/drive/MyDrive/efficio_checkpoints/cifar10`. Without Drive, checkpoints last only as long as the Colab runtime.

In [ ]:
run = efficio.Run(budget="45m", checkpoint_dir="/content/efficio_checkpoints/cifar10")
model, optimizer = run.prepare(model, optimizer, scheduler)

## 4. Train

`run.loop` yields batches already on the GPU and skips the batches done before a resume. `run.autocast()` runs the forward pass in fp16 or bf16, `run.backward` scales the loss for fp16, and `run.step()` steps the optimizer and the scheduler, saves periodically and stops when the budget is nearly used.

In [ ]:
for x, y in run.loop(train_loader, epochs=EPOCHS):
    with run.autocast():
        loss = loss_fn(model(x), y)
    run.backward(loss)
    run.step()

if run.should_stop:
    print("Stopped early and saved a checkpoint. Run the notebook again to continue.")

## 5. Evaluate

`run.evaluate_mode()` switches the model to eval mode, turns off gradients and uses the same mixed precision.

In [ ]:
correct = total = 0
with run.evaluate_mode():
    for x, y in test_loader:
        correct += (model(x).argmax(1) == y).sum().item()
        total += len(y)
print(f"test accuracy: {100 * correct / total:.2f}%")

## 6. Report

The report shows where the time went (setup, training, waiting for data), the average GPU utilization, the optimizations Efficio applied and tips for your setup.

In [ ]:
run.report()

## Resume after a disconnect

If the session ends or the budget runs out, run all cells again. `run.prepare` finds the latest checkpoint in `checkpoint_dir` and training continues from the same epoch and batch, with the same shuffle order. With checkpoints on Google Drive this also works in a new session.